# From reads to variants: human data

**Purpose.** Take raw sequencing reads all the way to called variants, so that the file
formats stop being black boxes. Every later exercise starts from one of the files you
make here.

**What you will do**
 - look at a FASTQ file and run FastQC on it
 - map the reads to a reference with `bwa mem`
 - convert, sort and index the alignment (SAM -> BAM)
 - inspect the alignment with `samtools tview` and `mpileup`, and look at the depth
 - call variants with `bcftools` and read the VCF

**The data.** **NA19238**, one individual from the **1000 Genomes Project**, population
**YRI** (Yoruba in Ibadan, Nigeria). Paired-end Illumina reads from **chromosome 21**,
171,880 read pairs of 100 bp, sequenced at low depth. The reference is human chromosome
21 only (46.9 Mb), so that mapping finishes in about a minute.

 - File formats (FASTQ, SAM/BAM, VCF)
 - Mapping (single-end, paired-end) NGS data to a reference sequence
 - Read flags
 - VERY IMPORTANT, you need to identify the 'pipe' button on your computer '|'.

## Environment setup

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data and software live
DATA=/course/data/current_data/NGSintro/human
SOFTWARE=/course/data/current_data/NGSintro/software

# where you will do the exercise
WORK_DIR=$HOME/ngs_intro_human

# input files
NA19238_1_FQ=$DATA/NA19238.YRI.low_coverage.chr21_1.fq.gz
NA19238_2_FQ=$DATA/NA19238.YRI.low_coverage.chr21_2.fq.gz
CHR21=$DATA/chr21.fa.gz

# programs
PICARD=$SOFTWARE/picard.jar
FASTQC=fastqc

# the same files as they are named inside WORK_DIR, and the sample name
# used for every file you create below
FQ1=$(basename $NA19238_1_FQ)
FQ2=$(basename $NA19238_2_FQ)
REF=$(basename $CHR21)
SAMPLE=NA19238

# the chromosome in the reduced reference
CHROM=chr21

# make WORK_DIR readable by the R and python cells further down
mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.ngs_intro_human_workdir

echo --programs that are installed:--
which samtools
which bwa
which angsd
which bcftools
which $FASTQC
ls $PICARD

echo --Datasets that will be used--
echo pair of fastQ files
ls $NA19238_1_FQ
ls $NA19238_2_FQ

echo reference genome
ls $CHR21


First make a folder for the exercise and add symbolic links to the reference genomes and the fastQ files

In [ ]:
# enter the folder that was created in the first cell
cd $WORK_DIR

#make links to files and add them to the folder
cp -sf  $NA19238_1_FQ .
cp -sf  $NA19238_2_FQ .
cp -sf  ${CHR21}* .

echo --- files in folder ---
ls


In [ ]:
# set up R working space
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.ngs_intro_human_workdir"))[1]
setwd(work_d)
getwd()


In [ ]:
# set up python working space
# the working directory was set in the first cell of the notebook
import os
work_d = open(os.path.expanduser("~/.ngs_intro_human_workdir")).read().strip()
os.chdir(work_d)
print(os.getcwd())


## Mapping one reduced genome
![ngs_files1](https://popgen.dk/albrecht/open/ngs_files1.png)
In this exercise you will align a fastq file using bwa and generate a SAM file.

Due to the computational time we have created a reduced genome from one of the individuals from the 1000 Genomes pilot project. The individual, NA19238, has been sequenced using Illumina short-read sequencing. For this exercise we have created a reduced reference genome consisting only of chromosome 21, the smallest human chromosome, and reduced the sequencing data to reads that will likely map to chromosome 21 within the first 15Mb of the chromosome.

The fastQ file NA19238.YRI.low_coverage.chr21_1.fq.gz has variable name with *_1.fq.gz, which is the first read of the read pair.

Before we start mapping we want to perform some QC of the data.

# Step 1: FastQ file and QC
![ngs_files2](https://popgen.dk/albrecht/open/ngs_files2.png)
### Viewing the input files

View the fastq file (NA19238.YRI.low_coverage.chr21_1.fq.gz) using the head command and identify the reads and quality scores (ignore the Broken pipe warning).


In [ ]:
# -n determines the number of lines printed
gunzip -c $FQ1 | head -n 12


Identify the read names, the sequence, the separator line, and the base quality scores in the FASTQ output above. Fill in the ???? below for one complete read.
<code>
?????        @read_name
?????        ACTG...
?????        +
?????        quality_scores
</code>

**Questions**
 - Each read takes up four lines. Which line holds the bases, and which holds their quality scores?
 - The sequence line and the quality line are the same length. Why must that be true?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/fastq_format.json')


The below command count the number of lines in the file
 - How many lines do you have ????
 - How many Reads in the data ????
 - is the number of lines the same in the 2 fastQ files ???? ( modify the code below to see the number of lines in the other file)

In [ ]:
gunzip -c $FQ2 |  wc -l

 - How many lines do you have?
 - How many reads are in the data? (there are four lines per read)
 - Is the number of lines the same in the two FASTQ files?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/fastq_counts_human.json')


View the reference fasta file (chr21.fa.gz) using the head command. The first many bases of the reference genome is all Ns ( unknown bases).
First view the first bases of chr21 then try to view other parts. You can modify the below uncommented code below



In [ ]:
# first 20 lines
gunzip -c $REF | head -n 20

# view another part of the reference (uncomment and modify below)
# gunzip -c $REF | head -n 200 | tail -n 20

**Questions**
 - The first line starts with `>`. What does that line tell you?
 - Why are there long stretches of `N` at the start of chromosome 21?
 - A FASTA file has no quality scores, while a FASTQ file does. Why does the reference not need them?

# step 1: FastQ file and QC
![ngs_files2](https://popgen.dk/albrecht/open/ngs_files2.png)

#### FastQC

Let's see if there are any issues with the sequencing reads.

In [ ]:
$FASTQC --nogroup $FQ1

echo ---- fastQC has created this file ----
ls *html


To view the report, switch to the main browser tab for the Jupyter notebook. Enter the exercise folder and find the html file. Click on the file to open the FastQC report.

It will look something like the picture below.

![FastQC file](https://github.com/popgenDK/courses/blob/main/current_exercises/ngs/figures/fastqc_report_human.png?raw=true)

**Questions**
 - How long are the reads?
 - Does the base quality drop towards the end of the reads? Why does that happen with Illumina sequencing?
 - Did any module fail or raise a warning?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/fastqc_human.json')



## Aligning
![ngs_files3](https://popgen.dk/albrecht/open/ngs_files3.png)

- Mapping to a reference genome
- What is in the sam/bam file?
- Quality: Mapping vs Alignment
- Sequencing Depth

Align the reads using bwa. We use bwa in the exercises because it is fast and widely used. We first need to index the reference chromosome, followed by the actual aligning process. If should take around 1 min to finish. 


In [ ]:
## # we will use the prepared index files
# bwa index $REF

The index is already built here, so the command is commented out. Building it on a full human genome takes well over an hour.

**Question**
 - Why does bwa need an index of the reference before it can map anything?

Once the index is made, the second step is to map the reads. There are several ways to do this, but I suggest you use the bwa mem mode, which is the most commonly used these days. Again you can run it with no arguments to get info about how to use it. 

In [ ]:
# see options
bwa mem

**Questions**
 - Which option sets the number of threads?
 - Which three inputs does `bwa mem` need in order to map paired-end data?

The number of options may be a bit overwhelming, but you can run it with no additional options, although I suggest you add "-t 5" to run 5 threads if your computer has multiple cores. It reads the compressed fastq files directly, so you need not decompress them. By default the result comes on stdout (in the terminal), so you have to redirect to a file, like the below command. 
We also want to add a read group name with information about where the reads comes from. This is very useful if you have sequencing data from multiple libraries.  
Now try to align the data


In [ ]:
#align the data ( take ~ 1 min)
bwa mem -R '@RG\tID:foo\tSM:bar\tLB:library1' -t 5 $REF $FQ1 $FQ2  > $SAMPLE.sam

**Questions**
 - The output was sent to a file with `>`. What would have happened without it?
 - `-R '@RG\tID:foo\tSM:bar\tLB:library1'` adds a read group. Why does it matter to record which sample a read came from, once you analyse many individuals together?

Let's look at the generated sam file

In [ ]:
# view first 20 lines
head -n20 $SAMPLE.sam

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/sam_format.json')


You can read about the sam output here: https://bioinformatics-core-shared-training.github.io/cruk-summer-school-2017/Day1/Session5-alignedReads.html  

 - Identify the header and explain its contents. 
 - For the first read identify the following and fill in the (?????) below
     - the chromosome
     - the position of the first base of the read 
     - The mapping qualty
     - The alignment (cigar string)
     - the insert size (template length)
     - the read(the bases)
     - the base qualities
 <code>
 SRR794309.186	(the name of the read)
 99 			(FLAGS)
 chr21			(?????)
 16239093		(?????
 60				(?????
 100M			(?????
 =				(name of the mate is the same)
 16239300		(position of the mate)
 307			(?????)
 CCTTTTTATGGCTGAGTAGTATTCCACAGTTTCTTTACCCACTCCTTGATCAATAGGCACTTGGGTTGGTTCCACGATTTTGCATTTGTGAATTGTGTTG		(?????)
 CCCFFFFFHHHHHJJJFHIFHHJIJJJJJHIJJJJJIIJJJJJJJJJIHIIJJJJJJJJJJJJJJFHIJHFHHHHFFFDDEEEEEEDEACCEEECDCCDD		(?????)
 NM:i:0	MD:Z:100	MC:Z:100M	AS:i:100	XS:i:26	RG:Z:foo  (TAGS)
 </code>
 
 
 To understand the flags (second column in the sam format) you can type a flag into this page and get the meaning: https://broadinstitute.github.io/picard/explain-flags.html
 


Let's try to find the number of reads  in the samfile.


In [ ]:
wc -l $SAMPLE.sam 

**Questions**
 - Why is this not the same number as in the FASTQ file?
 - The count includes the header lines. How could you count only the alignment records?

Why is it not the same number as in the fastQ file?



Fortunately there are tools to handle sam files, which will make your life easier. We will use the samtools program. First, you often need the compressed version of the sam format, which is called bam. You use samtools view for converting between formats. BAM files facilitates random access to genomic regions, but this requires the file to be sorted and requires  an index this is generated using the command below.
Converting sam to bam is done like this:

In [ ]:
#sam to bam
samtools view -b $SAMPLE.sam > $SAMPLE.bam
#sort bam file
samtools sort -o $SAMPLE.sorted.bam $SAMPLE.bam
#index bam file
samtools index $SAMPLE.sorted.bam

#see sizes
echo --- files sizes ---
ls -lah $SAMPLE.sam $SAMPLE.bam $SAMPLE.sorted.bam

**Questions**
 - Compare the three file sizes printed above. Roughly how much smaller is the BAM than the SAM?
 - Is any information lost in the conversion?
 - Why did we have to sort the file before indexing it?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/sam_to_bam.json')

The bam file is a compressed version of sam, you can see it is about one-third of the sam file in size. 



We now have a functional alignment file that we can use for analysis. Let's first view the alignment at different part of the chromosome 21. We will use tview to extract alignment. The option -d -w print  150 bases of the alignment to the terminal

In [ ]:
samtools tview $SAMPLE.sorted.bam  -d T -w 100 -p $CHROM:10002000

In the above the lines are

Line1: The position on chromosome 21

Line2: The reference genome ( N if not provided)

Line3: The consensus sequence (If all reads have a G then the consensus is G)

Line4+:  (lines 4,5 ect) the reads alignment


- When looking at the region starting with position chr21:10002000 can you find a possible variable site?
- look at chr21:10028350. Do you think there are problems with the alignment at this position?
- look at chr21:10042151. is this a variable site or is there another likely explanation?

Let's try to add the reference genome to make it esiaer to see the sequencing error and variable sites

In [ ]:
samtools tview $SAMPLE.sorted.bam  -d T -w 100 -p $CHROM:10042151 $REF

 - How many likely variable sites can you see?
 - Is it possible that both of the first two sites (sites 10042151 and 10042152) are heterozygous sites?
 - Have a look at the region starting with 9719896. Do you think the variable sites in this region are reliable (why/why not)?
 
 
 Another way to look at the genome is by generating a [pileup](http://samtools.sourceforge.net/samtools.shtml) format

In [ ]:
# see first 20 sites where there is data
samtools mpileup $SAMPLE.sorted.bam  | head -n 20


Each line is a position with data.
 - When is this a format particularly useful?
 
 
 From the pileup it is easily to get the sequencing depth distribution

In [ ]:
samtools mpileup $SAMPLE.sorted.bam | cut -f4 | sort -n | uniq -c >dep2
cat dep2

**Question**
 - The left column is the number of sites and the right one is the depth. Are positions with **zero** reads listed here? What does that mean for the numbers you are about to plot?


the left column is the number of sites and the right is the depth. 

View the distribution for this individuals using the following R command


In [ ]:
depth <- read.table("dep2")
d <- 1:15 #chosen depths to plot

barplot(depth[d+1,1],names=d,xlab="sequencing depth",ylab="Number of sites with sequencing depth ",col="mistyrose")


**Questions**
 - What is the most common sequencing depth?
 - At this depth, how confident would you be calling a heterozygous genotype from a single read?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/depth.json')

 - How do you think the depth will affect genotype and variant calling?
 
 First let's view the mpileup with the reference

In [ ]:
samtools mpileup -f $REF $SAMPLE.sorted.bam | head -n 20

 - Can you see the difference compared to not using the reference genome. 
 - Can you identify a heterozygous site? (e.g. position 9719896)
 
 
 ### Variant calling 
 ![ngs_files4](https://popgen.dk/albrecht/open/ngs_files4.png)

 - Pileup -> variant

  Let's create a VCF file for the first couple of MB of chr21. This is done based on the mpileup. There will be much more information tomorrow about how the calling is done using genotype likelihoods. However, before doing so we should remove duplicated reads ( read with the same starting points) as they are likely PCR duplicate

In [ ]:
## remove duplicates
samtools rmdup -s $SAMPLE.sorted.bam $SAMPLE.md.bam

## call variants
bcftools mpileup -Ou -f $REF $SAMPLE.md.bam | bcftools call -mv -Ov -o $SAMPLE.vcf

**Questions**
 - The VCF was made from `NA19238.md.bam`, the file with duplicates removed, not from the sorted BAM. Why?
 - `bcftools call` was run with `-mv`. What would the file look like without the `-v`?

Let's have a look at the VCF file

In [ ]:
head -n 50 $SAMPLE.vcf 


 The header of the VCF contains meta information about what it in the file.
In the body of the file
 - Identify the position, the reference allele and the alternative allele of the file.
 - Identify the depth of each position
 - Find a tri-allelic site. Do you believe that it is truly triallelic?
 - How many sites are called as variable?
 
 
 
 # Bonus exercise (Only do this part if you have finished the rest) 
 ## Bonus exercise 1 -  duplicated reads using Picardtools
 
 bwa actually fills in the mate information, but not all aligners do that, so we can run picard tools to fill in the mate information and sort the file according to position. We will output the file in the binary version of SAM which is BAM

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/vcf.json')

In [ ]:

java -jar $PICARD FixMateInformation INPUT=$SAMPLE.sam \
OUTPUT=id.fixmate.srt.bam SORT_ORDER=coordinate

**Question**
 - Picard was given the SAM file and wrote a BAM file sorted by coordinate. Which two things did it change in one step?

View the header of the BAM file

In [ ]:
samtools view -H id.fixmate.srt.bam 

**Question**
 - Which lines describe the reference sequences, and which describe the programs that have touched the file?

picard didn't update the PG flag, so let us update the header information so that we have documented how we modified the file.

In [ ]:

(samtools view -H id.fixmate.srt.bam;echo -e "@PG\tID:fixmate\tPN:fixmate\tVN:2.60\tCL:stuff" ) >newhd
samtools reheader newhd id.fixmate.srt.bam > id.fixmate.srt2.bam


**Question**
 - What was added to the header by the command above, and why is it good practice to record it?

 - Validate that the header in file id.fixmate.srt2.bam  has been updated

In [ ]:
samtools view -H id.fixmate.srt2.bam 

**Question**
 - Compare this header with the one you printed before. Which line is new?

Now mark duplicates using picard

In [ ]:

java -jar $PICARD MarkDuplicates I=id.fixmate.srt2.bam \
O=id.fixmate.srt.md.bam  M=metrics;


 - Did picard update the PG flag of the header?
 - Did picard update anything else in the header?

NB you can view the header of a bamfile using 'samtools view -H'




In [ ]:
samtools view -H id.fixmate.srt.md.bam



## Bonus exercise 2 - clean you bam files using the FLAGS column

The second column in the SAM format is the very important FLAG. This will tell you about the state of the paired-end mapping, QC duplicates etc.


  
Using the samtools -F/-f you can discard/include flags that fulfill certain patterns. See http://broadinstitute.github.io/picard/explain-flags.html .

  1. How many reads have we marked as duplicate in the final file.
  2. How many properly mapped read pairs do we have? (Where both reads map to the same chr etc).
  3. How many mapped reads do we have ?
  4. How many unmapped reads do we have ?
  5. Find the distribution of the RNAMES of the unmapped reads!?

 Run the following command one at a time by uncommenting them

In [ ]:



#samtools view -f 1024 id.fixmate.srt.md.bam|wc -l
#samtools view -f 2 id.fixmate.srt.md.bam|wc -l
#samtools view -F 4 id.fixmate.srt.md.bam|wc -l
#samtools view -f 4 id.fixmate.srt.md.bam|wc -l
# samtools view -f 4 id.fixmate.srt.md.bam|cut -f3|sort -n |uniq -c




**Questions**
 - Uncomment the lines above one at a time and run them. How many reads are unmapped?
 - How many are marked as duplicates?
 - `-f 4` and `-F 4` give different counts. What is the relationship between the two numbers?

Compare with "samtools flagstat" command 


In [ ]:
samtools flagstat id.fixmate.srt.md.bam


**Questions**
 - What fraction of the reads mapped?
 - What fraction were properly paired?
 - Do the numbers agree with what you counted by hand with `-f` and `-F`?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/flags.json')

Make a new bamfile, where you keep only the reads where both ends map, and filter out those with a mapping quality below 10, and removing duplicates


In [ ]:
samtools view -f 2 -F 1024 id.fixmate.srt.md.bam -q 10 >new.bam

**Questions**
 - How many reads are left in `new.bam` compared with the file you started from?
 - Each of the three filters (`-f 2`, `-F 1024`, `-q 10`) removes a different kind of read. Which one removed the most?

---

**You have now been through the whole pipeline: FASTQ → SAM → BAM → VCF.**
For each of the three file formats, try to say in one sentence what it stores and what one line in it represents.